In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import folium
from sklearn.neighbors import BallTree
from geopy.geocoders import Nominatim

print("Libraries loaded successfully!")


Libraries loaded successfully!


In [2]:
# Load the crime dataset
crime_data = pd.read_csv('crime.csv')

# Display the first few rows of the dataset
crime_data.head()


,nm_pol,murder,rape,gangrape,robbery,theft,assualt murders,sexual harassement,totarea,totalcrime,long,lat,crime/area,area
0,CHITRANJAN PARK,2,6,1,35,442,19,7,2659329.537,512,77.24920,28.53632,192.529731,2.659330
1,DABRI,8,28,0,79,240,26,16,3401013.428,397,77.08600,28.61268,116.729912,3.401013
2,MALVIYA NAGAR,3,28,1,33,694,63,15,1379853.572,837,77.20418,28.52989,606.586102,1.379854
3,CHANDNI MAHAL,1,8,1,23,529,19,7,5570696.132,588,77.23608,28.64361,105.552338,5.570696
4,MODEL TOWN,0,4,1,45,393,9,14,2689157.085,466,77.19369,28.70257,173.288501,2.689157


In [3]:
# Check the columns
print(crime_data.columns)

# Function to extract coordinates from 'crime/area' column
def extract_coordinates(loc):
    try:
        parts = loc.split(',')
        if len(parts) == 2: 
            latitude = float(parts[0].strip())
            longitude = float(parts[1].strip())
            return latitude, longitude
        else:
            return None, None  
    except:
        return None, None  

# Apply coordinate extraction and handle missing data
crime_data[['latitude', 'longitude']] = crime_data['crime/area'].apply(
    lambda loc: pd.Series(extract_coordinates(str(loc)))
)
crime_data = crime_data.dropna(subset=['latitude', 'longitude'])

# Check for missing values
print(crime_data['crime/area'].isnull().sum())
crime_data.head()


Index(['nm_pol', 'murder', 'rape', 'gangrape', 'robbery', 'theft',
       'assualt murders', 'sexual harassement', 'totarea', 'totalcrime',
       'long', 'lat', 'crime/area', 'area'],
      dtype='object')
0


,nm_pol,murder,rape,gangrape,robbery,theft,assualt murders,sexual harassement,totarea,totalcrime,long,lat,crime/area,area,latitude,longitude


In [4]:
# Select only necessary columns
crime_data = crime_data[['latitude', 'longitude']]

# Ensure latitude and longitude are numeric
crime_data['latitude'] = pd.to_numeric(crime_data['latitude'], errors='coerce')
crime_data['longitude'] = pd.to_numeric(crime_data['longitude'], errors='coerce')
crime_data = crime_data.dropna(subset=['latitude', 'longitude'])

# Convert to radians for haversine-based calculations
crime_data[['latitude', 'longitude']] = np.radians(crime_data[['latitude', 'longitude']])

# Check data integrity
print(crime_data.head())
print(crime_data.info())


Empty DataFrame
Columns: [latitude, longitude]
Index: []
<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   latitude   0 non-null      float64
 1   longitude  0 non-null      float64
dtypes: float64(2)
memory usage: 0.0 bytes
None


In [5]:
# Define a starting point and destination
starting_location = (19.0760, 72.8777)  # Example: Mumbai
destination = (18.5204, 73.8567)        # Example: Pune

# Route coordinates
route = [
    starting_location,
    ((starting_location[0] + destination[0]) / 2, (starting_location[1] + destination[1]) / 2),
    destination
]

# Convert route points to radians
route = [np.radians(coord) for coord in route]
print("Route defined:", route)


Route defined: [array([0.33293901, 1.27195582]), array([0.32809048, 1.2804992 ]), array([0.32324196, 1.28904259])]


In [6]:
# Function to calculate crime risk along the route
def calculate_crime_risk(route, crime_data, radius=0.5):
    crime_tree = BallTree(crime_data[['latitude', 'longitude']].values, metric='haversine')
    total_risk = 0
    
    for point in route:
        distances, indices = crime_tree.query_radius([point], r=radius / 6371, return_distance=False)
        total_risk += len(indices[0])
    
    return total_risk

# Compute risk score
risk_score = calculate_crime_risk(route, crime_data)
print(f"Risk Score for the route: {risk_score}")


ValueError: Found array with 0 sample(s) (shape=(0, 2)) while a minimum of 1 is required.

In [8]:
print(crime_data.head())
print(crime_data.info())

Empty DataFrame
Columns: [latitude, longitude]
Index: []
<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   latitude   0 non-null      float64
 1   longitude  0 non-null      float64
dtypes: float64(2)
memory usage: 0.0 bytes
None


In [9]:
from sklearn.neighbors import BallTree
import numpy as np

def calculate_crime_risk(route, crime_data, radius=0.5):
    # Check if crime_data is valid
    if crime_data.empty or not {'latitude', 'longitude'}.issubset(crime_data.columns):
        raise ValueError("Crime data is empty or does not contain 'latitude' and 'longitude' columns.")
    
    # Convert latitude and longitude to radians for haversine metric
    crime_coords = np.radians(crime_data[['latitude', 'longitude']].values)
    crime_tree = BallTree(crime_coords, metric='haversine')
    total_risk = 0

    for point in route:
        point_rad = np.radians([point])  # Convert route point to radians
        indices = crime_tree.query_radius(point_rad, r=radius / 6371, return_distance=False)
        total_risk += len(indices[0])
    
    return total_risk


In [15]:
import pandas as pd

# Example crime_data DataFrame
crime_data = pd.DataFrame({
    'latitude': [37.7749, 37.7849, 37.7949],
    'longitude': [-122.4194, -122.4294, -122.4394]
})

# Example route (list of latitude and longitude points)
route = [(37.7749, -122.4194), (37.7849, -122.4294)]

# Calculate risk score
try:
    risk_score = calculate_crime_risk(route, crime_data)
    print(f"Risk Score for the route: {risk_score}")
except ValueError as e:
    print(f"Error: {e}")


Risk Score for the route: 2


In [16]:
from sklearn.neighbors import BallTree
import numpy as np
import pandas as pd

# Function to calculate crime risk along the route
def calculate_crime_risk(route, crime_data, radius=0.5):
    # Ensure crime data has the required columns
    if crime_data.empty or not {'latitude', 'longitude'}.issubset(crime_data.columns):
        raise ValueError("Crime data is empty or missing 'latitude' and 'longitude' columns.")
    
    # Convert latitude and longitude to radians
    crime_coords = np.radians(crime_data[['latitude', 'longitude']].values)
    crime_tree = BallTree(crime_coords, metric='haversine')
    total_risk = 0

    for point in route:
        point_rad = np.radians([point])  # Convert route point to radians
        indices = crime_tree.query_radius(point_rad, r=radius / 6371, return_distance=False)
        total_risk += len(indices[0])  # Indices contains a list of nearby points
    
    return total_risk

# Example crime data for Mumbai-Pune region
crime_data = pd.DataFrame({
    'latitude': [18.5204, 19.0760, 18.5937, 18.9402, 18.5601],
    'longitude': [73.8567, 72.8777, 73.7898, 72.8346, 73.8902]
})

# Example route from Mumbai to Pune (latitude, longitude pairs)
route = [
    (19.0760, 72.8777),  # Mumbai
    (18.9402, 72.8346),  # Navi Mumbai
    (18.5601, 73.8902),  # Near Lonavala
    (18.5204, 73.8567)   # Pune
]

# Calculate risk score
try:
    risk_score = calculate_crime_risk(route, crime_data)
    print(f"Risk Score for the Mumbai-Pune route: {risk_score}")
except ValueError as e:
    print(f"Error: {e}")


Risk Score for the Mumbai-Pune route: 4


In [17]:
# # Initialize a folium map at the midpoint of the route
# route_map = folium.Map(location=((starting_location[0] + destination[0]) / 2,
#                                  (starting_location[1] + destination[1]) / 2), zoom_start=7)

# # Add route markers
# for point in route:
#     folium.Marker(location=np.degrees(point), icon=folium.Icon(color="blue")).add_to(route_map)

# # Add crime locations
# for _, crime in crime_data.iterrows():
#     folium.CircleMarker(
#         location=np.degrees((crime['latitude'], crime['longitude'])),
#         radius=5,
#         color="red",
#         fill=True,
#         fill_color="red"
#     ).add_to(route_map)

# # Display the map
# route_map


In [22]:
pip install pandas numpy folium scikit-learn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [24]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
import folium
from IPython.display import display


In [25]:
crime_data = pd.DataFrame({
    'latitude': [18.5204, 19.0760, 18.5937, 18.9402, 18.5601],
    'longitude': [73.8567, 72.8777, 73.7898, 72.8346, 73.8902]
})

In [26]:
route = [
    (19.0760, 72.8777),  # Mumbai
    (18.9402, 72.8346),  # Navi Mumbai
    (18.5601, 73.8902),  # Near Lonavala
    (18.5204, 73.8567)   # Pune
]

In [27]:
def is_route_safe(route, crime_data, radius_km=5):
    # Build a BallTree for efficient spatial search
    crime_tree = BallTree(np.radians(crime_data[['latitude', 'longitude']].values), metric='haversine')
    
    safe_segments = []
    unsafe_segments = []
    
    # Check each segment of the route
    for i in range(len(route) - 1):
        start, end = route[i], route[i + 1]
        mid_point = [(start[0] + end[0]) / 2, (start[1] + end[1]) / 2]  # Midpoint for safety check
        mid_point_rad = np.radians([mid_point])  # Convert to radians for haversine
        
        indices = crime_tree.query_radius(mid_point_rad, r=radius_km / 6371)  # Query within the radius
        if len(indices[0]) > 0:
            unsafe_segments.append((start, end))  # Unsafe if any crime data point is nearby
        else:
            safe_segments.append((start, end))  # Safe otherwise
    
    return safe_segments, unsafe_segments


In [28]:
def visualize_route(safe_segments, unsafe_segments):
    # Create a folium map centered around Pune
    m = folium.Map(location=[18.5204, 73.8567], zoom_start=10)
    
    # Plot safe segments in green
    for segment in safe_segments:
        folium.PolyLine([segment[0], segment[1]], color="green", weight=5).add_to(m)
    
    # Plot unsafe segments in red
    for segment in unsafe_segments:
        folium.PolyLine([segment[0], segment[1]], color="red", weight=5).add_to(m)
    
    return m


In [29]:
safe_segments, unsafe_segments = is_route_safe(route, crime_data)

In [30]:
route_map = visualize_route(safe_segments, unsafe_segments)


In [31]:
from IPython.display import IFrame
route_map.save("route_safety_map.html")  # Save map to an HTML file
display(IFrame("route_safety_map.html", width=800, height=600))


In [32]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
import folium
from IPython.display import display, IFrame

In [33]:
def is_route_safe(route, crime_data, radius_km=5):
    # Build a BallTree for efficient spatial search
    crime_tree = BallTree(np.radians(crime_data[['latitude', 'longitude']].values), metric='haversine')
    
    safe_segments = []
    unsafe_segments = []
    
    for i in range(len(route) - 1):
        start, end = route[i], route[i + 1]
        mid_point = [(start[0] + end[0]) / 2, (start[1] + end[1]) / 2]  # Midpoint for safety check
        mid_point_rad = np.radians([mid_point])
        
        indices = crime_tree.query_radius(mid_point_rad, r=radius_km / 6371)
        if len(indices[0]) > 0:
            unsafe_segments.append((start, end))
        else:
            safe_segments.append((start, end))
    
    return safe_segments, unsafe_segments

In [34]:
def generate_alternative_route(unsafe_segment):
    start, end = unsafe_segment
    # Mock alternative route: Deviate slightly from the unsafe segment
    mid_point_alt = [(start[0] + end[0]) / 2 + 0.01, (start[1] + end[1]) / 2 + 0.01]
    return [start, mid_point_alt, end]


In [35]:
def visualize_route_with_alternatives(safe_segments, unsafe_segments):
    m = folium.Map(location=[18.5204, 73.8567], zoom_start=10)
    
    # Plot safe segments in green
    for segment in safe_segments:
        folium.PolyLine([segment[0], segment[1]], color="green", weight=5).add_to(m)
    
    # Plot unsafe segments in red and their alternatives in blue
    for segment in unsafe_segments:
        folium.PolyLine([segment[0], segment[1]], color="red", weight=5).add_to(m)
        alt_route = generate_alternative_route(segment)
        folium.PolyLine(alt_route, color="blue", weight=5, dash_array="5, 10").add_to(m)
    
    return m

In [36]:
safe_segments, unsafe_segments = is_route_safe(route, crime_data)


In [37]:
route_map = visualize_route_with_alternatives(safe_segments, unsafe_segments)


In [38]:
route_map.save("route_with_alternatives.html")
display(IFrame("route_with_alternatives.html", width=800, height=600))